# Ticker Price Targets vs Price From Financial Modeling Prep

This notebook pulls the latest consensus price target snapshot for any ticker from FMP's `price-target-consensus` endpoint, then reconstructs a historical target band from FMP's analyst target history so we can compare price target trends against the stock's price.

Change `TICKER` in the first code cell and rerun the notebook. Inputs like `BRK\\B`, `BRK/B`, and `BRK.B` are normalized automatically for FMP.

Note: FMP's stable `price-target-consensus` endpoint returns the current snapshot only. The historical high, low, median, and consensus series below are reconstructed from FMP's `price-target-news` history using a trailing 365-day window of analyst targets.

In [ ]:
from pathlib import Path
import os

import pandas as pd
import plotly.graph_objects as go
import requests


TICKER = "AAPL"  # Change this to any supported ticker and rerun.
ROLLING_WINDOW_DAYS = 365
BASE_URL = "https://financialmodelingprep.com/stable"


def normalize_symbol_input(raw_symbol: str) -> str:
    symbol = raw_symbol.strip().upper()
    for separator in ("\\", "/", "."):
        symbol = symbol.replace(separator, "-")
    return symbol


SYMBOL = normalize_symbol_input(TICKER)


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / ".env").exists():
            return candidate
    raise FileNotFoundError("Could not find the repo .env file.")


def load_key_from_dotenv() -> str:
    dotenv_path = find_repo_root(Path.cwd()) / ".env"
    for raw_line in dotenv_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        name, value = line.split("=", 1)
        if name.strip() == "FMP_API_KEY":
            return value.strip().strip("\"'")
    raise RuntimeError("FMP_API_KEY was not found in .env")


def get_api_key() -> str:
    return os.getenv("FMP_API_KEY") or load_key_from_dotenv()


def request_json(endpoint: str, **params):
    response = requests.get(
        f"{BASE_URL}/{endpoint}",
        params={**params, "apikey": get_api_key()},
        timeout=30,
    )
    payload = response.json()

    if response.status_code != 200:
        raise RuntimeError(f"FMP request failed with status {response.status_code}: {payload}")

    if isinstance(payload, dict) and payload.get("Error Message"):
        raise RuntimeError(payload["Error Message"])

    if isinstance(payload, dict) and "value" in payload:
        return payload["value"]

    return payload


def fetch_all_price_target_news(symbol: str, page_size: int = 50) -> pd.DataFrame:
    rows = []
    page = 0

    while True:
        payload = request_json("price-target-news", symbol=symbol, page=page, limit=page_size)
        if not payload:
            break

        rows.extend(payload)
        if len(payload) < page_size:
            break

        page += 1

    target_news = pd.DataFrame(rows)
    if target_news.empty:
        raise RuntimeError(f"FMP returned no price target history for {symbol}")

    target_news["publishedDate"] = pd.to_datetime(target_news["publishedDate"], utc=True).dt.tz_convert(None)
    target_news["targetDate"] = target_news["publishedDate"].dt.normalize()
    target_news["targetValue"] = pd.to_numeric(
        target_news["adjPriceTarget"].fillna(target_news["priceTarget"]),
        errors="coerce",
    )
    target_news = target_news.dropna(subset=["targetValue"]).drop_duplicates(
        subset=["publishedDate", "analystCompany", "targetValue", "newsURL"]
    )
    return target_news.sort_values("publishedDate").reset_index(drop=True)


def build_historical_target_band(price_df: pd.DataFrame, target_news: pd.DataFrame, window_days: int) -> pd.DataFrame:
    rows = []
    window = pd.Timedelta(days=window_days)

    for price_date in price_df["date"]:
        active_targets = target_news.loc[
            (target_news["targetDate"] <= price_date)
            & (target_news["targetDate"] > price_date - window),
            "targetValue",
        ]

        if active_targets.empty:
            continue

        rows.append(
            {
                "date": price_date,
                "targetLow": active_targets.min(),
                "targetHigh": active_targets.max(),
                "targetMedian": active_targets.median(),
                "targetConsensus": active_targets.mean(),
                "targetCount": active_targets.count(),
            }
        )

    return pd.DataFrame(rows)


In [ ]:
quote_snapshot = request_json("quote", symbol=SYMBOL)
company_name = quote_snapshot[0].get("name", SYMBOL) if quote_snapshot else SYMBOL
chart_label = f"{company_name} ({SYMBOL})" if company_name != SYMBOL else SYMBOL

consensus_snapshot = pd.DataFrame(request_json("price-target-consensus", symbol=SYMBOL))
target_news = fetch_all_price_target_news(SYMBOL)
price_history = pd.DataFrame(request_json("historical-price-eod/light", symbol=SYMBOL))

price_history["date"] = pd.to_datetime(price_history["date"])
price_history["price"] = pd.to_numeric(price_history["price"], errors="coerce")
price_history = (
    price_history.loc[price_history["date"] >= target_news["targetDate"].min(), ["date", "price"]]
    .sort_values("date")
    .reset_index(drop=True)
)

consensus_snapshot


In [ ]:
target_band = build_historical_target_band(price_history, target_news, ROLLING_WINDOW_DAYS)
plot_df = price_history.merge(target_band, on="date", how="left")

latest_reconstructed = plot_df.dropna(subset=["targetConsensus"]).iloc[-1]
latest_snapshot = consensus_snapshot.iloc[0]

comparison = pd.DataFrame(
    [
        {
            "series": f"Trailing {ROLLING_WINDOW_DAYS}-day reconstructed band",
            "date": latest_reconstructed["date"].date(),
            "targetLow": round(latest_reconstructed["targetLow"], 2),
            "targetMedian": round(latest_reconstructed["targetMedian"], 2),
            "targetConsensus": round(latest_reconstructed["targetConsensus"], 2),
            "targetHigh": round(latest_reconstructed["targetHigh"], 2),
            "targetCount": int(latest_reconstructed["targetCount"]),
        },
        {
            "series": "Latest FMP price-target-consensus snapshot",
            "date": pd.Timestamp.today().date(),
            "targetLow": latest_snapshot["targetLow"],
            "targetMedian": latest_snapshot["targetMedian"],
            "targetConsensus": latest_snapshot["targetConsensus"],
            "targetHigh": latest_snapshot["targetHigh"],
            "targetCount": None,
        },
    ]
)

comparison


In [ ]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=plot_df["date"],
        y=plot_df["price"],
        name=f"{SYMBOL} close",
        line={"color": "#e5eefc", "width": 2.5},
    )
)

fig.add_trace(
    go.Scatter(
        x=plot_df["date"],
        y=plot_df["targetConsensus"],
        name=f"Consensus target ({ROLLING_WINDOW_DAYS}d trailing)",
        line={"color": "#60a5fa", "width": 2.5},
    )
)

fig.add_trace(
    go.Scatter(
        x=plot_df["date"],
        y=plot_df["targetMedian"],
        name=f"Median target ({ROLLING_WINDOW_DAYS}d trailing)",
        line={"color": "#fbbf24", "width": 2.5},
    )
)

fig.add_trace(
    go.Scatter(
        x=plot_df["date"],
        y=plot_df["targetHigh"],
        name=f"High target ({ROLLING_WINDOW_DAYS}d trailing)",
        line={"color": "#4ade80", "dash": "dash"},
    )
)

fig.add_trace(
    go.Scatter(
        x=plot_df["date"],
        y=plot_df["targetLow"],
        name=f"Low target ({ROLLING_WINDOW_DAYS}d trailing)",
        line={"color": "#f87171", "dash": "dash"},
        fill="tonexty",
        fillcolor="rgba(96, 165, 250, 0.14)",
    )
)

fig.add_trace(
    go.Scatter(
        x=[plot_df["date"].iloc[-1]],
        y=[latest_snapshot["targetConsensus"]],
        mode="markers+text",
        name="Latest consensus API snapshot",
        marker={"color": "#38bdf8", "size": 11, "symbol": "diamond"},
        text=[f"API snapshot: {latest_snapshot['targetConsensus']:.2f}"],
        textfont={"color": "#dbeafe"},
        textposition="top center",
    )
)

fig.update_layout(
    title=f"{chart_label} price vs analyst price target band",
    template="plotly_dark",
    paper_bgcolor="#020817",
    plot_bgcolor="#0f172a",
    font={"color": "#e2e8f0"},
    hovermode="x unified",
    hoverlabel={"bgcolor": "#0f172a", "font_color": "#e2e8f0"},
    legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0},
    xaxis_title="Date",
    yaxis_title="USD per share",
    autosize=True,
    height=680,
    margin={"l": 60, "r": 30, "t": 90, "b": 60},
)

fig.update_xaxes(showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True)
fig.update_yaxes(showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True)

fig.show(config={"responsive": True, "displaylogo": False})


## DCF Notebook

> The DCF workflow that used to live below this chart has been moved into `DCF vs Price - FMP.ipynb` in the same `Valuation` folder. Use that notebook for the FMP DCF snapshot, annual backfilled DCF build, quarterly TTM DCF build, and the annual-vs-quarterly comparison.